# Named Entity Recognition med spaCy

## Syfte och avgränsning

Syftet är att undersöka hur spaCys svenska modeller identifierar personer, organisationer och platser i svensk text. Jag jämför modellerna small, medium och large och undersöker hur de känner igen företagsnamn.

Jag testar även om medium-modellen kan förbättras genom finjustering med en liten mängd egen träningsdata. Arbetet är avgränsat till Named Entity Recognition och fokuserar inte på andra NLP-uppgifter.

NER är relevant inom Data Science eftersom stora mängder ostrukturerad text annars är svåra att analysera automatiskt. Ett företag skulle kunna använda NER för att bearbeta tusentals nyhetsartiklar eller pressmeddelanden och identifiera vilka företag, personer och platser som nämns.
I Volvo-texten kan modellen t.ex. hitta Volvo Car Group, Trafikverket, Erik Coelingh och Göteborg. Entiteterna kan sedan sparas som strukturerad data i en tabell och användas för att undersöka vilka organisationer som nämns tillsammans, vilka platser som förekommer mest.

## Viktiga begrepp och etiketter

Named Entity Recognition (NER) innebär att identifiera namngivna entiteter i text och klassificera dem i olika kategorier. Det kan till exempel vara personer, organisationer, platser och tidsuttryck.

Den svenska spaCy-modellen använder bland annat följande etiketter:

- ``PRS`` - personnamn t.ex. Jack Olofsson.
- ``ORG`` - organisationer, företag eller myndigheter, t.ex. Spotify eller Skatteverket.
- ``LOC`` - geografiska platser t.ex. Göteborg eller Sverige.
- ``OBJ`` - namngivna objekt. I mina tester användes etiketten bland annat för Volvo. Modellen verkade tolka namnet som ett bilmärke istället för som en organisation.
- ``TME`` - tidsuttryck t.ex. "igår" eller "nästa år"
- ``WRK`` - namn på t.ex. verk, produkter eller projekt. I mina resultat klassificerades projektnamnet "Drive Me" som WRK.

Etiketterna är modellens förutsägelser. Att modellen sätter en etikett betyder inte automatiskt att etiketten är korrekt.

## Första testet

Jag börjarde med en enkel mening som innehåller en person, en organisation och en plats. Målet är att se vilka entiteter spaCy hittar och vilka etiketter de får.

In [1]:
import spacy

nlp_sm = spacy.load("sv_core_news_sm")

text = "Jack Olofsson studerar på EC utbildning i Göteborg."

doc = nlp_sm(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

Jack Olofsson PRS
Göteborg LOC


In [2]:
ec_testtexter = [
    "Jack Olofsson studerar på EC-Utbildning i Göteborg.",
    "Företaget EC Utbildning har en skola i Göteborg.",
    "Skolan EC Utbildning har en skola i Göteborg."
]

for text in ec_testtexter:
    doc = nlp_sm(text)

    print(f"\nText: {text}")

    for ent in doc.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Jack Olofsson studerar på EC-Utbildning i Göteborg.
'Jack Olofsson' - PRS
'Göteborg' - LOC

Text: Företaget EC Utbildning har en skola i Göteborg.
'Göteborg' - LOC

Text: Skolan EC Utbildning har en skola i Göteborg.
'Skolan EC Utbildning' - ORG
'Göteborg' - LOC


### Resultat

I första testet hittade modellen ``Jack Olofsson`` som person och ``Göteborg`` som plats men den hittade inte EC-Utbildning som en organisation.

Jag testade sedan meningen "Företaget EC Utbildning har en skola i Göteborg" men modellen hittade fortfarande inte EC Utbildning som en organisation.

I meningen med "Företaget EC Utbildning" hittas ingen organisation alls. När meningen istället börjar med "Skolan EC Utbildning" identifieras hela uttrycket som en organisation. Det visar att sammanhanget påverkar modellens resultat och att modellen även kan göra fel på var en entitet börjar och slutar.

In [3]:
testtexter = [
    "Företaget Volvo har ett kontor i Göteborg.",
    "Företaget Volvo AB har ett kontor i Göteborg.",
    "Jag köpte en Volvo igår.",
    "Företaget Spotify har ett kontor i Stockholm.",
    "Företaget Spotify AB har ett kontor i Stockholm.",
    "Företaget Klarna har ett kontor i Stockholm.",
    "Företaget Klarna AB har ett kontor i Stockholm.",
    "Företaget EC Utbildning har en skola i Göteborg.",
    "Företaget EC Utbildning AB har en skola i Göteborg."
]

for text in testtexter:
    doc = nlp_sm(text)

    print("\nText:", text)

    for ent in doc.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Företaget Volvo har ett kontor i Göteborg.
'Volvo' - OBJ
'Göteborg' - LOC

Text: Företaget Volvo AB har ett kontor i Göteborg.
'Volvo AB' - OBJ
'Göteborg' - LOC

Text: Jag köpte en Volvo igår.
'Volvo' - OBJ

Text: Företaget Spotify har ett kontor i Stockholm.
'Stockholm' - LOC

Text: Företaget Spotify AB har ett kontor i Stockholm.
'Spotify AB' - ORG
'Stockholm' - LOC

Text: Företaget Klarna har ett kontor i Stockholm.
'Stockholm' - LOC

Text: Företaget Klarna AB har ett kontor i Stockholm.
'Klarna AB' - ORG
'Stockholm' - LOC

Text: Företaget EC Utbildning har en skola i Göteborg.
'Göteborg' - LOC

Text: Företaget EC Utbildning AB har en skola i Göteborg.
'Utbildning AB' - ORG
'Göteborg' - LOC


### Resultat

Här hade modellen svårt att identifiera flera av organisationerna. Volvo hittades som en entitet men klassificerades som OBJ både med och utan "AB". Spotify och Klarna hittades inte utan "AB", men när "AB" lades till identifierades både Spotify AB och Klarna AB som ORG. Jag ville även testa om modellen klassificerar Volvo olika beroende på sammanhanget eftersom Volvo både kan syfta på själva företaget och bilmärket.

EC Utbildning hittades inte utan "AB". När "AB" lades till identifierade modellen däremot "Utbildning AB" som ORG men missade "EC". Det visar att "AB" verkar hjälpa modellen att känna igen vissa organisationer men att den fortfarande kan göra fel på var entiteten börjar och slutar.

Modellen lyckades samtidigt hitta både Stockholm och Göteborg som LOC i alla meningar. I just dessa exempel verkar modellen därför ha lättare att känna igen platser än organisationer.

## Jämförelse mellan small och medium

Jag jämförde small och medium modellen på samma meningar för att se om modellstorleken påverkade vilka entiteter som hittades och hur de klassificerades.

In [4]:
nlp_md = spacy.load("sv_core_news_md")

In [5]:
import pandas as pd

testtexter = [
    "Företaget Volvo har ett kontor i Göteborg.",
    "Företaget Spotify har ett kontor i Stockholm.",
    "Företaget Klarna har ett kontor i Stockholm.",
    "Företaget EC Utbildning har en skola i Göteborg."
]

resultat = []

for text in testtexter:
    doc_sm = nlp_sm(text)
    doc_md = nlp_md(text)

    sm_entities = [(ent.text, ent.label_) for ent in doc_sm.ents]
    md_entities = [(ent.text, ent.label_) for ent in doc_md.ents]

    resultat.append({
        "Text": text,
        "SM": sm_entities,
        "MD": md_entities
    })

df_resultat = pd.DataFrame(resultat)
df_resultat


,Text,SM,MD
0,Företaget Volvo har ett kontor i Göteborg.,"[(Volvo, OBJ), (Göteborg, LOC)]","[(Volvo, ORG), (Göteborg, LOC)]"
1,Företaget Spotify har ett kontor i Stockholm.,"[(Stockholm, LOC)]","[(Stockholm, LOC)]"
2,Företaget Klarna har ett kontor i Stockholm.,"[(Stockholm, LOC)]","[(Stockholm, LOC)]"
3,Företaget EC Utbildning har en skola i Göteborg.,"[(Göteborg, LOC)]","[(EC, ORG), (Göteborg, LOC)]"


### Resultat

Medium-modellen gav ett bättre resultat i några av testerna. Medium-modellen lyckades nu att klassificera Volvo som ``ORG`` medan small-modellen fortfarande klassificerade Volvo som ``OBJ``.

Medium-modellen hittade även EC som ``ORG`` i "EC Utbildning", men den tog inte med hela namnet. Båda modellerna missade Spotify och Klarna i dessa meningar. Resultatet visar därför att den större modellen kan ge annorlunda och ibland bättre resultat, men att den fortfarande kan göra fel.

### Visualisering med displaCy

SpaCy innehåller verktyget displaCy som kan markera entiteter direkt i texten. Visualiseringen gör det enklare att se både vilken etikett en entitet har fått.

In [6]:
from spacy import displacy

text = "Jack Olofsson studerar på EC Utbildning i Göteborg."
doc = nlp_md(text)

displacy.render(doc, style="ent", jupyter=True)

### Test av personnamn

Jag testade även medium-modellen på olika typer av namn. Syftet var att undersöka om modellen fungerade lika bra för alla namn och om bindestreck påverkade resultatet.

In [7]:
test_personer = [
    "Jan-Åke Svensson arbetar i Stockholm.",
    "Mohammed Ali arbetar i Göteborg.",
    "Nour El-Refai studerar i Malmö.",
    "Jag träffade Erik igår.",
    "Jag träffade Jack igår."
]

for text in test_personer:
    doc = nlp_md(text)

    print(f"\nText: {text}")
    for ent in doc.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Jan-Åke Svensson arbetar i Stockholm.
'Jan-Åke Svensson' - PRS
'Stockholm' - LOC

Text: Mohammed Ali arbetar i Göteborg.
'Göteborg' - LOC

Text: Nour El-Refai studerar i Malmö.
'Malmö' - LOC

Text: Jag träffade Erik igår.
'Erik' - PRS
'igår' - TME

Text: Jag träffade Jack igår.
'Jack' - PRS


### Resultat

Modellen lyckades identifiera flera namn som PRS men missade Mohammed Ali och Nour El-Refai. Det visar att modellen inte alltid känner igen namn även när de står i ett sammanhang där det är tydligt att det handlar om en person.

Modellen hittade också platserna korrekt. I meningen "Jag träffade Erik igår" identifierades "igår" som TME men i meningen "Jag träffade Jack igår" identifierades inte "igår" alls. Det visar att samma ord kan få olika resultat beroende på resten av meningen och att modellen inte alltid är konsekvent.

## Jämförelse mellan small, medium och large


In [8]:
nlp_lg = spacy.load("sv_core_news_lg")


for text in testtexter:
    print(f"\nText: {text}")

    print("SM:")
    doc_sm = nlp_sm(text)
    for ent in doc_sm.ents:
        print(f"'{ent.text}' - {ent.label_}")

    print("MD:")
    doc_md = nlp_md(text)
    for ent in doc_md.ents:
        print(f"'{ent.text}' - {ent.label_}")

    print("LG:")
    doc_lg = nlp_lg(text)
    for ent in doc_lg.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Företaget Volvo har ett kontor i Göteborg.
SM:
'Volvo' - OBJ
'Göteborg' - LOC
MD:
'Volvo' - ORG
'Göteborg' - LOC
LG:
'Volvo' - ORG
'Göteborg' - LOC

Text: Företaget Spotify har ett kontor i Stockholm.
SM:
'Stockholm' - LOC
MD:
'Stockholm' - LOC
LG:
'Stockholm' - LOC

Text: Företaget Klarna har ett kontor i Stockholm.
SM:
'Stockholm' - LOC
MD:
'Stockholm' - LOC
LG:
'Stockholm' - LOC

Text: Företaget EC Utbildning har en skola i Göteborg.
SM:
'Göteborg' - LOC
MD:
'EC' - ORG
'Göteborg' - LOC
LG:
'Göteborg' - LOC


In [9]:
for text in test_personer:
    doc = nlp_lg(text)

    print(f"\nText: {text}")
    for ent in doc.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Jan-Åke Svensson arbetar i Stockholm.
'Jan-Åke Svensson' - PRS
'Stockholm' - LOC

Text: Mohammed Ali arbetar i Göteborg.
'Mohammed Ali' - PRS
'Göteborg' - LOC

Text: Nour El-Refai studerar i Malmö.
'Malmö' - LOC

Text: Jag träffade Erik igår.
'Erik' - PRS
'igår' - TME

Text: Jag träffade Jack igår.
'Jack' - PRS
'igår' - TME


### Resultat

Resultaten visar att en större modell inte automatiskt ger bättre resultat.

När jag testade organisationer klassificerade medium- och large-modellen Volvo som ORG, medan small-modellen klassificerade Volvo som OBJ. Alla tre modellerna missade däremot Spotify och Klarna. För EC Utbildning hittade medium-modellen "EC" som ORG, medan small- och large-modellen inte hittade organisationen alls.


När jag jämförde medium och large-modellen på namnen fungerade large-modellen bättre i flera av dessa exempel. Den identifierade bland annat Mohammed Ali som ``PRS`` och hittade "igår" som ``TME`` i båda de sista meningarna. Samtidigt missade den fortfarande Nour El-Refai.

Mina tester visar därför att ingen av modellerna verkar vara bäst i alla situationer. Medium-modellen fungerade bättre i vissa tester med organisationer medan large-modellen fungerade bättre i flera tester med personer.

## Test med en verklig text

Nu testar jag modellerna på ett utdrag från ett pressmeddelande från Volvo Cars. Texten innehåller bland annat namn, platser, organisationer och namn på projekt. Det gör att jag kan se hur modellerna fungerar på en längre och mer verklig text.

Källa: https://www.volvocars.com/se/media/press-releases/CA90C6F155FAEA0A/

In [10]:
with open("data/volvo_pressmeddelande.txt", "r", encoding="utf-8") as file:
    verklig_text = file.read()

print(verklig_text[:150])

Volvo Car Groups första självkörande bilar provas på allmänna vägar kring Göteborg

Volvo Car Groups banbrytande projekt "Drive Me", med 100 självköra


In [11]:
resultat_verklig_text = []

for namn, modell in [
    ("SM", nlp_sm),
    ("MD", nlp_md),
    ("LG", nlp_lg)
]:
    # första rundan i loopen
    #   = nlp_sm(verklig_text)
    doc = modell(verklig_text)

    for ent in doc.ents:
        # lägger till en ny dict för varje entitet som hittas
        resultat_verklig_text.append({
            #    key: value
            "Modell": namn,
            "Entitet": ent.text,
            "Etikett": ent.label_
        })

df_verklig_text = pd.DataFrame(resultat_verklig_text)
df_verklig_text

,Modell,Entitet,Etikett
0,SM,Volvo Car Groups,OBJ
1,SM,Göteborg,LOC
2,SM,Volvo Car Groups,OBJ
3,SM,Göteborg,LOC
4,SM,Erik Coelingh,PRS
5,SM,Technical Specialist,PRS
6,SM,Volvo Car Group,OBJ
7,SM,Volvo Car Group,OBJ
8,SM,Trafikverket,LOC
9,SM,Transportstyrelsen,ORG


### Resultat

I den längre texten blev skillnaderna mellan modellerna tydligare. Alla tre modellerna hittade Erik Coelingh som person och Göteborg som plats.

Volvo Car Group klassificerades däremot som OBJ i alla tre modellerna, trots att det är en organisation. Trafikverket klassificerades som LOC i small-modellen men som ORG i medium- och large-modellen. Även Göteborgs Stad klassificerades olika mellan modellerna.

Det visar att modellerna kan ge olika resultat på samma text och att en entitet inte alltid får rätt etikett även när namnet är tydligt för en människa.

### Återanvändbar funktion

För att undvika att upprepa samma kod flera gånger skapade jag funktionen ``find_entities`` i ``src/utils.py``. Funktionen tar emot en text och en spaCy-modell och returnerar de entiteter som modellen hittar.

In [12]:
from src.utils import find_entities

test_text = "Erik arbetar på Volvo i Göteborg."


df_test_text = pd.DataFrame(find_entities(test_text, nlp_md))
df_test_text

,Entity,Label
0,Erik,PRS
1,Volvo,ORG
2,Göteborg,LOC


## Finjustering av NER-modellen

I tidigare tester hade modellen svårt att känna igen vissa företag som organisationer, till exempel Spotify, Klarna och EC Utbildning. Därför vill jag testa att finjustera den redan tränade medium-modellen med egna exempel där jag själv har markerat vilka delar av texten som är ORG och LOC.

Innan jag börjar träningen skriver jag ut varje markerad del av texten. Detta gör jag för att kontrollera att det verkligen pekar på rätt ord. Om positionerna är fel får modellen fel träningsdata vilket kommer påverka resultatet.

In [13]:
train_data = [
    (
        "Företaget Volvo har ett kontor i Göteborg.",
        {"entities": [(10, 15, "ORG"), (33, 41, "LOC")]}
    ),
    (
        "Företaget Volvo AB har ett kontor i Göteborg.",
        {"entities": [(10, 18, "ORG"), (36, 44, "LOC")]}
    ),
    (
        "Företaget Spotify har ett kontor i Stockholm.",
        {"entities": [(10, 17, "ORG"), (35, 44, "LOC")]}
    ),
    (
        "Företaget Spotify AB har ett kontor i Stockholm.",
        {"entities": [(10, 20, "ORG"), (38, 47, "LOC")]}
    ),
    (
        "Företaget Klarna har ett kontor i Stockholm.",
        {"entities": [(10, 16, "ORG"), (34, 43, "LOC")]}
    ),
    (
        "Företaget Klarna AB har ett kontor i Stockholm.",
        {"entities": [(10, 19, "ORG"), (37, 46, "LOC")]}
    ),
    (
        "Företaget EC Utbildning har en skola i Göteborg.",
        {"entities": [(10, 23, "ORG"), (39, 47, "LOC")]}
    ),
    (
        "Företaget EC Utbildning AB har en skola i Göteborg.",
        {"entities": [(10, 26, "ORG"), (42, 50, "LOC")]}
    )
]

# train_data är en lista av tuples. Varje tuple innehåller två delar:
# 1. text = själva meningen
# 2. annotations = en dictionary med facit för vilka delar som är entiteter
for text, annotations in train_data:
    print(f"\nText: {text}")

    # annotations["entities"] är en lista av tuples.
    # Varje tuple innehåller startindex, slutindex och etikett.
    # De packas upp till variablerna start, end och label.
    for start, end, label in annotations["entities"]:
        print(f"{text[start:end]} -> {label}")


Text: Företaget Volvo har ett kontor i Göteborg.
Volvo -> ORG
Göteborg -> LOC

Text: Företaget Volvo AB har ett kontor i Göteborg.
Volvo AB -> ORG
Göteborg -> LOC

Text: Företaget Spotify har ett kontor i Stockholm.
Spotify -> ORG
Stockholm -> LOC

Text: Företaget Spotify AB har ett kontor i Stockholm.
Spotify AB -> ORG
Stockholm -> LOC

Text: Företaget Klarna har ett kontor i Stockholm.
Klarna -> ORG
Stockholm -> LOC

Text: Företaget Klarna AB har ett kontor i Stockholm.
Klarna AB -> ORG
Stockholm -> LOC

Text: Företaget EC Utbildning har en skola i Göteborg.
EC Utbildning -> ORG
Göteborg -> LOC

Text: Företaget EC Utbildning AB har en skola i Göteborg.
EC Utbildning AB -> ORG
Göteborg -> LOC


### Pipeline

Jag vill även se vilka delar som ingår i spaCy-pipelinen. Under finjusteringen kommer jag bara att träna NER-delen och tillfälligt stänga av de andra delarna.

In [14]:
print(nlp_md.pipe_names)

['tok2vec', 'tagger', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


In [15]:
import random

from spacy.training import Example


# Laddar den färdigtränade svenska medium-modellen eftersom det är denna jag vill finjustera
nlp2 = spacy.load("sv_core_news_md")


# Jag vill bara träna "ner" av modellen. De andra delarna, t.ex. tagger och parser stänger jag av tillfälligt
# Sparar alla pipe_names i en lista som inte är "ner"
other_pipes = [pipe for pipe in nlp2.pipe_names if pipe != "ner"]


# Stänger tillfälligt av alla pipeline-delar i other_pipes så att bara NER-delen är aktiv under träningen
# Efter with-blocket återgår pipelinen till sitt vanliga läge igen
with nlp2.select_pipes(disable=other_pipes):

    # Fortsätter träningen från modellens nuvarande vikter som används senare i nlp2.update()
    optimizer = nlp2.resume_training()

    # Hur många gånger modellen ska gå igenom hela train_data
    epochs = 10

    # Sätter seed för att få samma blandning av träningsdatan mellan körningar
    random.seed(37)

    for epoch in range(epochs):

        # Blandar ordningen på träningsmeningarna inför varje varv
        random.shuffle(train_data)

        # Skapar en tom dictionary för denna epoken. spaCy fyller den med loss när modellen uppdateras
        # Jag använder den för att kunna följa om loss minskar mellan epokerna. Dictionaryn nollställs i början av varje ny epok
        losses = {}

        # Går igenom en träningsmening i taget. Varje tuple i train_data packas upp i text och annotations
        for text, annotations in train_data:

            # Gör om textsträngen till ett spaCy Doc-objekt. make_doc() delar upp den i tokens men kör inte resten av pipelinen som t.ex. "ner"
            doc = nlp2.make_doc(text)

            # Skapar ett träningsobjekt där spaCy får både texten och det rätta svaret från annotations att jämföra mot
            example = Example.from_dict(doc, annotations)

            # Modellen gör en förutsägelse, jämför med facit och justerar sina vikter utifrån felet
            nlp2.update(
                [example],
                sgd=optimizer,
                drop=0.2,
                losses=losses
            )

        # Visar hur stort felet blev efter varje epoch
        print(f"Epoch {epoch + 1}, Loss: {float(losses["ner"]):.4f}")


# Sparar den finjusterade modellen i en egen mapp
nlp2.to_disk("custom_ner_model")

Epoch 1, Loss: 2.7798
Epoch 2, Loss: 0.0616
Epoch 3, Loss: 0.0230
Epoch 4, Loss: 0.0000
Epoch 5, Loss: 0.0178
Epoch 6, Loss: 0.0000
Epoch 7, Loss: 0.0000
Epoch 8, Loss: 0.0000
Epoch 9, Loss: 0.0000
Epoch 10, Loss: 0.0000


### Ladda den finjusterade modellen

In [16]:
# Laddar den sparade, finjusterade modellen från mappen
trained_nlp = spacy.load("custom_ner_model")


# Nya meningar som modellen inte tränades exakt på
test_text_custom_nlp = [
    "Organisationen Volvo har ett kontor i Göteborg.",
    "Organisationen Spotify har ett kontor i Stockholm.",
    "EC Utbildning erbjuder en ny utbildning i Helsingborg."
]


# Testar den finjusterade modellen på nya meningar
for text in test_text_custom_nlp:

    # Skickar texten genom den finjusterade spaCy-modellen
    doc = trained_nlp(text)
    print(f"\nText: {text}")

    # Går igenom alla entiteter som modellen hittade i texten
    for ent in doc.ents:
        # Skriver ut själva entiteten och vilken etikett den fick
        print(f"'{ent.text}' - {ent.label_}")


Text: Organisationen Volvo har ett kontor i Göteborg.
'Volvo' - ORG
'Göteborg' - LOC

Text: Organisationen Spotify har ett kontor i Stockholm.
'Spotify' - ORG
'Stockholm' - LOC

Text: EC Utbildning erbjuder en ny utbildning i Helsingborg.
'EC Utbildning' - ORG
'Helsingborg' - LOC


### Resultat

Den sparade modellen kunde laddas och användas på nya meningar. Den identifierade Volvo och EC Utbildning som ORG och Göteborg, Stockholm och Helsingborg som LOC.

För Spotify tog modellen däremot med ordet "Organisationen" som en del av entiteten och klassificerade hela uttrycket "Organisationen Spotify" som ORG. Det visar att modellen kan hitta rätt typ av entitet men fortfarande göra fel på var entiteten börjar och slutar.

## Test på den verkliga texten

Nu testar jag modellerna på samma utdrag ur Volvo Cars pressmeddelande.

In [17]:
from src.utils import compare_models


models_to_compare = [
    ("MD original", nlp_md),
    ("MD finjusterad", trained_nlp)
]

df_comparison = pd.DataFrame(
    compare_models(verklig_text, models_to_compare)
)

df_comparison

,Model,Entity,Label
0,MD original,Volvo Car Groups,OBJ
1,MD original,Göteborg,LOC
2,MD original,Volvo Car Groups,OBJ
3,MD original,Göteborg,LOC
4,MD original,Autopilot,ORG
5,MD original,Erik Coelingh,PRS
6,MD original,Technical Specialist,ORG
7,MD original,Volvo Car Group,OBJ
8,MD original,Volvo Car Group,OBJ
9,MD original,Trafikverket,ORG


### Resultat
När jag jämförde den ursprungliga medium-modellen med den finjusterade modellen såg jag en tydlig förbättring för Volvo Car Group. Originalmodellen klassificerade Volvo Car Group som OBJ medan den finjusterade modellen klassificerade namnet som ORG.

Samtidigt behöll modellen flera tidigare resultat t.ex. Göteborg som LOC och Erik Coelingh som PRS. I vissa träningskörningar identifierade den finjusterade modellen även "Drive Me" som ``WRK``, men resultatet förekom inte i alla körningar.

Resultatet visar att finjusteringen verkar ha förbättrat modellen för vissa organisationer utan att den i den här texten tappade tidigare kunskap.

## Test på nya meningar



In [18]:
unseen_text = [
    "Spotify AB öppnar ett nytt kontor i Göteborg nästa år.",
    "Klarna AB anställer fler personer i Stockholm.",
    "Volvo presenterade en ny satsning under måndagen.",
    "Jack Olofsson läser en kurs hos EC Utbildning i Göteborg."
]


for text in unseen_text:
    print(f"\nText: {text}")

    print("MD original:")
    doc_original = nlp_md(text)

    for ent in doc_original.ents:
        print(f"'{ent.text}' - {ent.label_}")

    print("MD finjusterad:")
    doc_trained = trained_nlp(text)

    for ent in doc_trained.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Spotify AB öppnar ett nytt kontor i Göteborg nästa år.
MD original:
'Göteborg' - LOC
'nästa år' - TME
MD finjusterad:
'Spotify AB' - ORG
'Göteborg' - LOC

Text: Klarna AB anställer fler personer i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Klarna AB' - ORG
'Stockholm' - LOC

Text: Volvo presenterade en ny satsning under måndagen.
MD original:
'Volvo' - OBJ
MD finjusterad:
'Volvo' - ORG

Text: Jack Olofsson läser en kurs hos EC Utbildning i Göteborg.
MD original:
'Jack Olofsson' - PRS
'EC' - ORG
'Göteborg' - LOC
MD finjusterad:
'Jack Olofsson' - PRS
'EC Utbildning' - ORG
'Göteborg' - LOC


### Resultat

Jag testade båda modellerna på nya meningar som inte fanns med i träningsdatan. Den finjusterade modellen identifierade Spotify AB och Klarna AB som ORG medan originalmodellen inte hittade dem alls. Volvo ändrades från OBJ till ORG och för EC Utbildning hittade den finjusterade modellen hela namnet istället för bara "EC".

I dessa tester verkar finjusteringen alltså ha förbättrat modellens förmåga att känna igen de organisationer som jag tränade den på. Samtidigt behöll modellen andra entiteter som platser, personer och tid.

## Test med andra företag



In [19]:
generalization_texts = [
    "Volumental utvecklar ny teknik för skobutiker i Stockholm.",
    "Volumental AB utvecklar ny teknik för skobutiker i Stockholm.",
    "Elastisys öppnar ett nytt kontor i Stockholm.",
    "Elastisys AB öppnar ett nytt kontor i Stockholm.",
    "Bower lanserar en ny tjänst i Sverige.",
    "Funnel anställer fler utvecklare i Stockholm."
]

for text in generalization_texts:
    print(f"\nText: {text}")

    print("MD original:")
    doc_original = nlp_md(text)

    for ent in doc_original.ents:
        print(f"'{ent.text}' - {ent.label_}")

    print("MD finjusterad:")
    doc_trained = trained_nlp(text)

    for ent in doc_trained.ents:
        print(f"'{ent.text}' - {ent.label_}")


Text: Volumental utvecklar ny teknik för skobutiker i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Stockholm' - LOC

Text: Volumental AB utvecklar ny teknik för skobutiker i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Volumental AB' - ORG
'Stockholm' - LOC

Text: Elastisys öppnar ett nytt kontor i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Stockholm' - LOC

Text: Elastisys AB öppnar ett nytt kontor i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Elastisys AB' - ORG
'Stockholm' - LOC

Text: Bower lanserar en ny tjänst i Sverige.
MD original:
'Sverige' - LOC
MD finjusterad:
'Bower' - ORG
'Sverige' - LOC

Text: Funnel anställer fler utvecklare i Stockholm.
MD original:
'Stockholm' - LOC
MD finjusterad:
'Funnel' - ORG
'Stockholm' - LOC


### Resultat

Jag testade även modellen på företag som inte fanns med i min egen träningsdata. Originalmodellen identifierade inget av företagen som organisationer.

Den finjusterade modellen identifierade Bower och Funnel som ORG. När jag testade Volumental och Elastisys utan "AB" identifierades inget av namnen som organisationer. När jag lade till "AB" identifierades däremot både Volumental AB och Elastisys AB som ORG.

Det tyder på att den finjusterade modellen har lärt sig vissa mönster som hjälper den att känna igen nya organisationer och att "AB" verkar vara en viktig ledtråd.

## Begränsningar

Träningsdatan består endast av åtta meningar och flera av meningarna har nästan samma struktur. Att loss snabbt sjunker till noll betyder inte att modellen fungerar bra på all svensk text. Det kan istället tyda på att modellen har lärt sig de få texterna väl.

Testerna bygger enbart på ett litet antal manuellt valda meningar. Jag har inte beräknat mått som precision, recall och F1. Resultaten ska därför ses som en mindre kvalitativ demonstration och inte som en fullständig utvärdering av modellen.

Träningen innehåller slumpmoment bland annat genom dropout (drop= 0.2). Jag märkte därför att resultatet kunde variera mellan träningskörningar. Exempelvis identifierades "Drive Me" som WRK i vissa körningar men inte i andra. Den lilla mängden träningsdata gör modellen känslig för sådana variationer.

## Slutsats från finjusteringen

I mina tester förbättrade finjusteringen modellens förmåga att känna igen flera organisationer. Den kunde efter finjusteringen identifiera Volvo, Spotify, Klarna och EC Utbildning bättre än originalmodellen.

Resultaten visar samtidigt att modellen fortfarande inte är helt säker. Hur ett namn skrivs och vilket sammanhang det finns i påverkar resultatet. Eftersom träningsdatan är väldigt liten går det inte att säga att modellen generellt har blivit bättre på att känna igen organisationer i text.